# Config

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

In [0]:
QUARTER = 'Q3'
YEAR = 2025
base_path = "abfss://cms-part-d@cmsraw.dfs.core.windows.net/q3-2025"

In [0]:
def plan_key_with_segment(df):
    return (df
      .withColumn("CONTRACT_ID", F.upper(F.trim(F.col("CONTRACT_ID"))))
      .withColumn("PLAN_ID", F.trim(F.col("PLAN_ID")))
      .withColumn("SEGMENT_ID", F.coalesce(F.trim(F.col("SEGMENT_ID")), F.lit("000")))
      .withColumn("PLAN_KEY", F.concat(F.col("CONTRACT_ID"), F.col("PLAN_ID"), F.col("SEGMENT_ID")))
    )

def plan_key_no_segment(df):
    return (df
      .withColumn("CONTRACT_ID", F.upper(F.trim(F.col("CONTRACT_ID"))))
      .withColumn("PLAN_ID", F.trim(F.col("PLAN_ID")))
      .withColumn("PLAN_KEY", F.concat(F.col("CONTRACT_ID"), F.col("PLAN_ID"), F.lit("000")))
    )


In [0]:

def add_ingest_meta(df, source_filename: str):
    return (df
            .withColumn("CONTRACT_YEAR", F.lit(YEAR))
            .withColumn("CONTRACT_QUARTER", F.lit(QUARTER))
            .withColumn("SOURCE_FILENAME", F.lit(source_filename))
            .withColumn("INGESTION_TS", F.current_timestamp())
            .withColumn("UPDATED_AT", F.current_timestamp()))

def add_record_hash(df, ignore_cols):
    ignore_cols = set(ignore_cols)
    cols = [c for c in df.columns if c not in ignore_cols]
    exprs = [F.coalesce(F.col(c).cast("string"), F.lit("∅")) for c in cols]
    return df.withColumn("RECORD_HASH", F.sha2(F.concat_ws("||", *exprs), 256))

def read_source(file_path, schema, encoding="utf-8"):
    file_name = file_path.split('/')[-1].split('.')[0]

    df = spark.read.format("csv")\
                            .options(header="true", delimiter="|", encoding="latin1")\
                            .schema(schema)\
                            .load(f"{base_path}/{file_name}.txt")

    # df = add_ingest_meta(df, file_name)
    # df = add_record_hash(df, ['CONTRACT_YEAR', 'CONTRACT_QUARTER', 'INGESTION_TS', 'UPDATED_AT', 'SOURCE_FILENAME'])
    return df


def insert_unique_rows(df, target_table):
    # spark.table(slv_table).createOrReplaceTempView("t")
    df.createOrReplaceTempView("s")
    spark.sql(f"""
        MERGE INTO {target_table} t
        USING s
        ON t.record_hash = s.record_hash
        WHEN NOT MATCHED THEN INSERT *
    """)

# Define schema

## Plan Information

In [0]:
plan_info_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("CONTRACT_NAME", T.StringType()),
    T.StructField("PLAN_NAME", T.StringType()),
    T.StructField("FORMULARY_ID", T.StringType()),
    T.StructField("PREMIUM", T.DoubleType()),
    T.StructField("DEDUCTIBLE", T.DoubleType()),
    T.StructField("MA_REGION_CODE", T.StringType()),
    T.StructField("PDP_REGION_CODE", T.StringType()),
    T.StructField("STATE", T.StringType()),
    T.StructField("COUNTY_CODE", T.StringType()),
    T.StructField("SNP", T.IntegerType()),
    T.StructField("PLAN_SUPPRESSED_YN", T.StringType()),
])


In [0]:
plan_info = read_source(f"{base_path}/plan_information_PPUF_2025Q1.txt", encoding='latin1')

plan_info = plan_info.withColumn(
                "MA_REGION_CODE",
                F.when(
                    F.trim(F.col("MA_REGION_CODE")) == "", None
                ).otherwise(F.col("MA_REGION_CODE")).cast("double")
            ).withColumn(
                "PDP_REGION_CODE",
                F.when(
                    F.trim(F.col("PDP_REGION_CODE")) == "", None
                ).otherwise(F.col("PDP_REGION_CODE")).cast("double")
            )

plan_info = add_ingest_meta(plan_info, source_filename='plan_information_PPUF_2025Q1')
plan_info = add_record_hash(plan_info, ['CONTRACT_YEAR', 'CONTRACT_QUARTER', 'INGESTION_TS', 'UPDATED_AT', 'SOURCE_FILENAME'])

In [0]:
insert_unique_rows(plan_info, 'cms_partd_bronze.cdc.cdc_plan_info')

In [0]:
display(plan_info)

## Basic Drugs Formulary

In [0]:
_schema = spark.sql("select * from cms_partd_bronze.raw.brz_plan_info limit 1").schema

In [0]:
basic_formulary_schema = T.StructType([
    T.StructField("FORMULARY_ID", T.StringType()),
    T.StructField("FORMULARY_VERSION", T.StringType()),
    T.StructField("CONTRACT_YEAR", T.StringType()),
    T.StructField("RXCUI", T.StringType()),
    T.StructField("NDC", T.StringType()),
    T.StructField("TIER_LEVEL_VALUE", T.DoubleType()),
    T.StructField("QUANTITY_LIMIT_YN", T.StringType()),
    T.StructField("QUANTITY_LIMIT_AMOUNT", T.StringType()),
    T.StructField("QUANTITY_LIMIT_DAYS", T.StringType()),
    T.StructField("PRIOR_AUTHORIZATION_YN", T.StringType()),
    T.StructField("STEP_THERAPY_YN", T.StringType()),
])


In [0]:
def tf_basic_formulary(df):
    df = (df
      .withColumn("FORMULARY_ID", F.trim(F.col("FORMULARY_ID")))
      .withColumn("RXCUI", F.trim(F.col("RXCUI")))
      .withColumn("NDC", F.trim(F.col("NDC")))
      .withColumn("FORM_DRUG_KEY", F.concat(F.col("FORMULARY_ID"), F.lit("_"), F.col("NDC")))
      .withColumn("TIER_LEVEL_VALUE", F.col("TIER_LEVEL_VALUE").cast("double"))
      .withColumn("QUANTITY_LIMIT_AMOUNT", F.col("QUANTITY_LIMIT_AMOUNT").cast("double"))
      .withColumn("QUANTITY_LIMIT_DAYS", F.col("QUANTITY_LIMIT_DAYS").cast("double"))
      .withColumn("HAS_QUANTITY_LIMIT", F.upper(F.trim(F.col("QUANTITY_LIMIT_YN"))) == F.lit("Y"))
      .withColumn("HAS_PRIOR_AUTH", F.upper(F.trim(F.col("PRIOR_AUTHORIZATION_YN"))) == F.lit("Y"))
      .withColumn("HAS_STEP_THERAPY", F.upper(F.trim(F.col("STEP_THERAPY_YN"))) == F.lit("Y"))
    )

    df = (df
      .withColumn("RESTRICTION_COUNT",
          F.col("HAS_QUANTITY_LIMIT").cast("int") +
          F.col("HAS_PRIOR_AUTH").cast("int") +
          F.col("HAS_STEP_THERAPY").cast("int")
      )
      .withColumn("IS_GENERIC_TIER", F.col("TIER_LEVEL_VALUE") <= 2)
      .withColumn("IS_BRAND_TIER", (F.col("TIER_LEVEL_VALUE") >= 3) & (F.col("TIER_LEVEL_VALUE") < 5))
      .withColumn("IS_SPECIALTY_TIER", F.col("TIER_LEVEL_VALUE") >= 5)
    )
    return df


## Excluded Drugs

In [0]:
excluded_drugs_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("RXCUI", T.StringType()),
    T.StructField("TIER", T.DoubleType()),
    T.StructField("QUANTITY_LIMIT_YN", T.StringType()),   # 0/1 per PDF
    T.StructField("QUANTITY_LIMIT_AMOUNT", T.StringType()),
    T.StructField("QUANTITY_LIMIT_DAYS", T.StringType()),
    T.StructField("PRIOR_AUTH_YN", T.StringType()),
    T.StructField("STEP_THERAPY_YN", T.StringType()),
    T.StructField("CAPPED_BENEFIT_YN", T.StringType()),
])


In [0]:
def tf_excluded_drugs(df):
    df = plan_key_no_segment(df)

    df = (df
      .withColumn("RXCUI", F.trim(F.col("RXCUI")))
      .withColumn("TIER", F.col("TIER").cast("double"))
      # QUANTITY_LIMIT_YN uses '0'/'1' in your script
      .withColumn("HAS_QUANTITY_LIMIT", F.trim(F.col("QUANTITY_LIMIT_YN")) == F.lit("1"))
      .withColumn("HAS_PRIOR_AUTH", F.upper(F.trim(F.col("PRIOR_AUTH_YN"))) == F.lit("Y"))
      .withColumn("HAS_STEP_THERAPY", F.upper(F.trim(F.col("STEP_THERAPY_YN"))) == F.lit("Y"))
      .withColumn("HAS_CAPPED_BENEFIT", F.upper(F.trim(F.col("CAPPED_BENEFIT_YN"))) == F.lit("Y"))
      .withColumn("COMPLETELY_EXCLUDED", F.col("TIER").isNull())
      .withColumn("CONDITIONALLY_COVERED", ~F.col("TIER").isNull())
    )
    return df


## Indication Based Coverage

In [0]:
indication_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("RXCUI", T.StringType()),
    T.StructField("DISEASE", T.StringType()),
])


In [0]:
def tf_indication_coverage(df):
    df = plan_key_no_segment(df)

    df = (df
      .withColumn("RXCUI", F.trim(F.col("RXCUI")))
      .withColumn("DISEASE", F.trim(F.col("DISEASE")))
      .withColumn("INDICATION_KEY", F.concat_ws("_", F.col("PLAN_KEY"), F.col("RXCUI"), F.col("DISEASE")))
    )

    # INDICATIONS_PER_DRUG: count per (PLAN_KEY, RXCUI)
    w = Window.partitionBy("PLAN_KEY", "RXCUI")
    df = df.withColumn("INDICATIONS_PER_DRUG", F.count(F.lit(1)).over(w))
    return df


## Beneficiary Cost

In [0]:
benef_cost_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("COVERAGE_LEVEL", T.IntegerType()),
    T.StructField("TIER", T.DoubleType()),
    T.StructField("DAYS_SUPPLY", T.IntegerType()),
    T.StructField("COST_TYPE_PREF", T.IntegerType()),
    T.StructField("COST_AMT_PREF", T.DoubleType()),
    T.StructField("COST_MIN_AMT_PREF", T.StringType()),
    T.StructField("COST_MAX_AMT_PREF", T.DoubleType()),
    T.StructField("COST_TYPE_NONPREF", T.IntegerType()),
    T.StructField("COST_AMT_NONPREF", T.DoubleType()),
    T.StructField("COST_MIN_AMT_NONPREF", T.StringType()),
    T.StructField("COST_MAX_AMT_NONPREF", T.DoubleType()),
    T.StructField("COST_TYPE_MAIL_PREF", T.IntegerType()),
    T.StructField("COST_AMT_MAIL_PREF", T.DoubleType()),
    T.StructField("COST_MIN_AMT_MAIL_PREF", T.StringType()),
    T.StructField("COST_MAX_AMT_MAIL_PREF", T.DoubleType()),
    T.StructField("COST_TYPE_MAIL_NONPREF", T.IntegerType()),
    T.StructField("COST_AMT_MAIL_NONPREF", T.DoubleType()),
    T.StructField("COST_MIN_AMT_MAIL_NONPREF", T.StringType()),
    T.StructField("COST_MAX_AMT_MAIL_NONPREF", T.DoubleType()),
    T.StructField("TIER_SPECIALTY_YN", T.StringType()),
    T.StructField("DED_APPLIES_YN", T.StringType()),
])


In [0]:
def tf_beneficiary_cost(df):
    df = plan_key_with_segment(df)

    # numeric casts
    df = (df
      .withColumn("COVERAGE_LEVEL", F.col("COVERAGE_LEVEL").cast("int"))
      .withColumn("TIER", F.col("TIER").cast("double"))
      .withColumn("DAYS_SUPPLY", F.col("DAYS_SUPPLY").cast("int"))
    )

    # COST_KEY exactly like pandas (tier as string -> e.g. "1.0")
    df = (df
      .withColumn("COST_KEY",
          F.concat(
            F.col("PLAN_KEY"),
            F.lit("_T"), F.col("TIER").cast("string"),
            F.lit("_C"), F.col("COVERAGE_LEVEL").cast("string"),
            F.lit("_D"), F.col("DAYS_SUPPLY").cast("string")
          )
      )
    )

    # parse MIN_AMT fields (coerce -> null if non-numeric)
    for ptype in ["PREF", "NONPREF", "MAIL_PREF", "MAIL_NONPREF"]:
        df = df.withColumn(f"COST_MIN_AMT_{ptype}", F.col(f"COST_MIN_AMT_{ptype}").cast("double"))

    # flags
    df = (df
      .withColumn("IS_SPECIALTY_TIER", F.upper(F.trim(F.col("TIER_SPECIALTY_YN"))) == "Y")
      .withColumn("DEDUCTIBLE_APPLIES", F.upper(F.trim(F.col("DED_APPLIES_YN"))) == "Y")
    )

    # mappings (same as your dicts)
    df = (df
      .withColumn("DAYS_SUPPLY_LABEL",
          F.when(F.col("DAYS_SUPPLY") == 1, "30_days")
           .when(F.col("DAYS_SUPPLY") == 2, "90_days")
           .when(F.col("DAYS_SUPPLY") == 4, "60_days")
           .otherwise("other")
      )
      .withColumn("COVERAGE_LEVEL_LABEL",
          F.when(F.col("COVERAGE_LEVEL") == 0, "pre_deductible")
           .when(F.col("COVERAGE_LEVEL") == 1, "initial_coverage")
           .when(F.col("COVERAGE_LEVEL") == 3, "catastrophic")
           .otherwise("other")
      )
    )

    # cost type labels
    def cost_type_label(c):
        return (F.when(F.col(c) == 0, "not_offered")
                 .when(F.col(c) == 1, "copay")
                 .when(F.col(c) == 2, "coinsurance")
                 .otherwise(None))

    for ptype in ["PREF", "NONPREF", "MAIL_PREF", "MAIL_NONPREF"]:
        df = df.withColumn(f"COST_TYPE_{ptype}_LABEL", cost_type_label(f"COST_TYPE_{ptype}"))

    # min/max across pharmacy types (uses COST_AMT_* columns in your script)
    cost_cols = ["COST_AMT_PREF", "COST_AMT_NONPREF", "COST_AMT_MAIL_PREF", "COST_AMT_MAIL_NONPREF"]
    df = (df
      .withColumn("MIN_COST", F.least(*[F.col(c).cast("double") for c in cost_cols]))
      .withColumn("MAX_COST", F.greatest(*[F.col(c).cast("double") for c in cost_cols]))
    )

    # BEST_PHARMACY_TYPE = idxmin equivalent
    min_cost = F.col("MIN_COST")
    df = (df
      .withColumn("BEST_PHARMACY_TYPE",
          F.when(F.col("COST_AMT_PREF").cast("double") == min_cost, "PREF")
           .when(F.col("COST_AMT_NONPREF").cast("double") == min_cost, "NONPREF")
           .when(F.col("COST_AMT_MAIL_PREF").cast("double") == min_cost, "MAIL_PREF")
           .when(F.col("COST_AMT_MAIL_NONPREF").cast("double") == min_cost, "MAIL_NONPREF")
           .otherwise(None)
      )
    )
    return df


## Insulin Beneficiary Cost

In [0]:
insulin_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("TIER", T.StringType()),                 # may be missing per PDF
    T.StructField("DAYS_SUPPLY", T.IntegerType()),
    T.StructField("COPAY_AMT_PREF_INSLN", T.DoubleType()),
    T.StructField("COPAY_AMT_NONPREF_INSLN", T.DoubleType()),
    T.StructField("COPAY_AMT_MAIL_PREF_INSLN", T.DoubleType()),
    T.StructField("COPAY_AMT_MAIL_NONPREF_INSLN", T.DoubleType()),
])


## Pricing

In [0]:
pricing_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("NDC", T.StringType()),
    T.StructField("DAYS_SUPPLY", T.IntegerType()),
    T.StructField("UNIT_COST", T.DoubleType()),
])


## Pharmacy Networks

In [0]:
pharm_net_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("PHARMACY_NUMBER", T.StringType()),
    T.StructField("PHARMACY_ZIPCODE", T.StringType()),
    T.StructField("PREFERRED_STATUS_RETAIL", T.StringType()),
    T.StructField("PREFERRED_STATUS_MAIL", T.StringType()),
    T.StructField("PHARMACY_RETAIL", T.StringType()),
    T.StructField("PHARMACY_MAIL", T.StringType()),
    T.StructField("IN_AREA_FLAG", T.IntegerType()),
    T.StructField("FLOOR_PRICE", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_30", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_60", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_90", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_30", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_60", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_90", T.DoubleType()),
])


In [0]:
file_name = 'pharmacy_networks_file_PPUF_2025Q3_part_6'

pharm_net = read_source(f"{base_path}/{file_name}.txt", schema=pharm_net_schema, encoding='utf-8')


pharm_net = add_ingest_meta(pharm_net, source_filename=file_name)
pharm_net = add_record_hash(pharm_net, ['CONTRACT_YEAR', 'CONTRACT_QUARTER', 'INGESTION_TS', 'UPDATED_AT', 'SOURCE_FILENAME'])

In [0]:
insert_unique_rows(pharm_net, 'cms_partd_bronze.cdc.cdc_pharmacy_network')

In [0]:
def tf_pharmacy_network(df):
    df = plan_key_with_segment(df)

    df = (df
      .withColumn("PHARMACY_NUMBER", F.trim(F.col("PHARMACY_NUMBER")))
      .withColumn("NETWORK_KEY", F.concat_ws("_", F.col("PLAN_KEY"), F.col("PHARMACY_NUMBER")))
      .withColumn("IS_PREFERRED_RETAIL", F.upper(F.trim(F.col("PREFERRED_STATUS_RETAIL"))) == "Y")
      .withColumn("IS_PREFERRED_MAIL", F.upper(F.trim(F.col("PREFERRED_STATUS_MAIL"))) == "Y")
      .withColumn("OFFERS_RETAIL", F.upper(F.trim(F.col("PHARMACY_RETAIL"))) == "Y")
      .withColumn("OFFERS_MAIL", F.upper(F.trim(F.col("PHARMACY_MAIL"))) == "Y")
      .withColumn("IS_IN_AREA", F.col("IN_AREA_FLAG").cast("int") == 1)
    )

    brand_cols = ["BRAND_DISPENSING_FEE_30","BRAND_DISPENSING_FEE_60","BRAND_DISPENSING_FEE_90"]
    gen_cols   = ["GENERIC_DISPENSING_FEE_30","GENERIC_DISPENSING_FEE_60","GENERIC_DISPENSING_FEE_90"]

    df = (df
      .withColumn("AVG_BRAND_FEE", (F.col(brand_cols[0]).cast("double")+F.col(brand_cols[1]).cast("double")+F.col(brand_cols[2]).cast("double"))/3)
      .withColumn("AVG_GENERIC_FEE", (F.col(gen_cols[0]).cast("double")+F.col(gen_cols[1]).cast("double")+F.col(gen_cols[2]).cast("double"))/3)
    )

    # PHARMACY_TYPE logic (same override order as your script)
    df = df.withColumn("PHARMACY_TYPE", F.lit("standard"))
    df = df.withColumn("PHARMACY_TYPE", F.when(F.col("IS_PREFERRED_RETAIL"), "preferred_retail").otherwise(F.col("PHARMACY_TYPE")))
    df = df.withColumn("PHARMACY_TYPE", F.when(F.col("IS_PREFERRED_MAIL") & F.col("OFFERS_MAIL"), "preferred_mail").otherwise(F.col("PHARMACY_TYPE")))

    return df


In [0]:
raw_cols = spark.sql("select * from cms_partd_bronze.raw.brz_pharmacy_network limit 1").columns

pharm_net = spark.sql(f"select * from cms_partd_bronze.cdc.cdc_pharmacy_network")

pharm_net = tf_pharmacy_network(pharm_net)


In [0]:
pharm_net.select(raw_cols).write.mode("overwrite").saveAsTable("cms_partd_bronze.raw.brz_pharmacy_network")

## Geographic Locator

In [0]:
geo_schema = T.StructType([
    T.StructField("COUNTY_CODE", T.StringType()),
    T.StructField("STATENAME", T.StringType()),
    T.StructField("COUNTY", T.StringType()),
    T.StructField("MA_REGION_CODE", T.StringType()),
    T.StructField("MA_REGION", T.StringType()),
    T.StructField("PDP_REGION_CODE", T.StringType()),
    T.StructField("PDP_REGION", T.StringType()),
])


# check

In [0]:
pharm_net_schema = T.StructType([
    T.StructField("CONTRACT_ID", T.StringType()),
    T.StructField("PLAN_ID", T.StringType()),
    T.StructField("SEGMENT_ID", T.StringType()),
    T.StructField("PHARMACY_NUMBER", T.StringType()),
    T.StructField("PHARMACY_ZIPCODE", T.StringType()),
    T.StructField("PREFERRED_STATUS_RETAIL", T.StringType()),
    T.StructField("PREFERRED_STATUS_MAIL", T.StringType()),
    T.StructField("PHARMACY_RETAIL", T.StringType()),
    T.StructField("PHARMACY_MAIL", T.StringType()),
    T.StructField("IN_AREA_FLAG", T.IntegerType()),
    T.StructField("FLOOR_PRICE", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_30", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_60", T.DoubleType()),
    T.StructField("BRAND_DISPENSING_FEE_90", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_30", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_60", T.DoubleType()),
    T.StructField("GENERIC_DISPENSING_FEE_90", T.DoubleType()),
])

In [0]:
phar_net = spark.sql("""select CONTRACT_ID
          ,PLAN_ID
          ,SEGMENT_ID
          ,PHARMACY_NUMBER
          ,PHARMACY_ZIPCODE
          ,PREFERRED_STATUS_RETAIL
          ,PREFERRED_STATUS_MAIL
          ,PHARMACY_RETAIL
          ,PHARMACY_MAIL
          ,IN_AREA_FLAG
          ,FLOOR_PRICE
          ,BRAND_DISPENSING_FEE_30
          ,BRAND_DISPENSING_FEE_60
          ,BRAND_DISPENSING_FEE_90
          ,GENERIC_DISPENSING_FEE_30
          ,GENERIC_DISPENSING_FEE_60
          ,GENERIC_DISPENSING_FEE_90
          ,2025 CONTRACT_YEAR
          ,'Q3' CONTRACT_QUARTER
          ,'pharmacy_networks_file_PPUF_2025Q3_part_1' SOURCE_FILENAME
          ,INGESTION_TS
          ,INGESTION_TS UPDATED_AT
          from cms_partd_bronze.raw.brz_pharmacy_network""")

phar_net = add_record_hash(phar_net, ['CONTRACT_YEAR', 'CONTRACT_QUARTER', 'INGESTION_TS', 'UPDATED_AT', 'SOURCE_FILENAME'])

In [0]:
phar_net.write.mode("overwrite").saveAsTable("cms_partd_bronze.cdc.cdc_pharmacy_network")